# Reto 2 · SECOP: RAG, MCP y memoria

**Taller 02.** Analista sobre 20 pliegos **simulados con fines educativos**. No representa una decisión jurídica ni un proceso de contratación real.

| Requisito | Evidencia |
|---|---|
| RAG sobre los 20 PDF | §2: índice, estadísticas, búsqueda citada y búsqueda vacía |
| Herramientas MCP por protocolo stdio | §3: cliente oficial, herramientas y llamada |
| Respuestas con fuente | §4: documentos y MCP |
| «No lo sé» cuando falta información | §2, §4 y evaluación §6 |
| Memoria entre sesiones | §5: dos instancias, persistencia, snapshot y olvido |
| Precisión de extracción | §6: 20 PDF × 14 campos contra clave |
| Coste, tokens, latencia y trazas | §4 y §6 |
| Comparación Claude / Ollama y decisiones | §6 y §7 |

El notebook usa `await` y se ejecuta en orden. La primera evaluación consume API; las siguientes reutilizan checkpoints. `--force` repite las llamadas pagadas.

## 1. Preparación

Desde la raíz: `uv sync`, preparar `.env` y ejecutar `uv run jupyter lab`. El kernel debe pertenecer al entorno del proyecto. Las claves se comprueban sin mostrarlas. Se mantiene `LANGSMITH_TRACING=true`.

**Tres componentes distintos:** sentence-transformers ejecuta localmente `paraphrase-multilingual-MiniLM-L12-v2` y genera vectores. **ChromaDB almacena y consulta esos vectores**; cuando recibe texto llama a la función de embeddings configurada, no genera los embeddings por sí mismo. El **LLM de chat** interpreta las preguntas y usa herramientas: Claude Haiku por API o Qwen mediante Ollama. Cambiar el LLM no cambia el espacio vectorial del corpus ni de la memoria.

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
import asyncio
import json
import sys
from dataclasses import asdict
from pathlib import Path
from tempfile import TemporaryDirectory
from IPython.display import Markdown, display
from secop_agent.config import PROJECT_ROOT, RESULTS_DIR, get_settings
os.chdir(PROJECT_ROOT)
settings = get_settings()
print({"project_root": str(PROJECT_ROOT), "model": settings.chat_model,
       "anthropic_key_present": bool(os.getenv("ANTHROPIC_API_KEY")),
       "langsmith_key_present": bool(os.getenv("LANGSMITH_API_KEY")),
       "langsmith_tracing": settings.tracing_enabled,
       "ollama_model": "ollama:qwen2.5:3b"})
assert settings.has_llm_credentials, "Configure ANTHROPIC_API_KEY in .env"
assert settings.tracing_enabled, "Enable LangSmith tracing in .env"


{'project_root': '/home/esteban/Documents/University/Tenth semester/IA/unit-iii/challenge-ii/reto2-secop-rag-mcp', 'model': 'anthropic:claude-haiku-5-5', 'anthropic_key_present': True, 'langsmith_key_present': True, 'langsmith_tracing': True, 'ollama_model': 'ollama:qwen2.5:3b'}


## 2. RAG documental

Los fragmentos respetan las secciones numeradas y conservan PDF, página, sección, referencia y entidad. Los identificadores filtran el proceso antes del ranking: veinte documentos de estructura parecida no se distinguen únicamente por similitud semántica.

Umbrales: score mínimo 0,40; cobertura léxica mínima 0,60; ranking 80% similitud coseno + 20% cobertura, con bonificación de 0,20 para título exacto. La calibración usó el mismo fixture de 24 preguntas: sus resultados son descriptivos, no una prueba independiente de generalización.

In [2]:
from collections import Counter
from secop_agent.rag import TenderRetriever, load_corpus
from secop_agent.rag.store import build_index
chunks = await asyncio.to_thread(load_corpus)
index_size = await asyncio.to_thread(build_index, chunks)
retriever = TenderRetriever()
print({"pdf_count": len(retriever.list_documents()), "chunks": index_size,
       "pages_with_chunk_starts": dict(Counter(chunk.page for chunk in chunks))})
assert len(retriever.list_documents()) == 20
hits = await asyncio.to_thread(retriever.search, "¿Cuál es el índice de liquidez mínimo de FDLS-LP-098-2019?")
for hit in hits:
    print(hit.citation(), f"score={hit.score:.3f}")
    print(hit.chunk.text)
assert hits
empty_hits = await asyncio.to_thread(retriever.search, "¿A qué distancia está la Luna de Júpiter?")
print("Resultado fuera del corpus:", empty_hits)
assert empty_hits == []


Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'pdf_count': 20, 'chunks': 157, 'pages_with_chunk_starts': {1: 99, 2: 58}}
[fuente: 11_FDLS_LP_098_2019.pdf, p. 2, §5 Capacidad financiera y organizacional (según RUP)] score=0.777
FDLS-LP-098-2019 · ALCALDÍA LOCAL DE SUMAPAZ · 5. Capacidad financiera y organizacional (según RUP)
Indicador Requisito
Índice de liquidez Mayor o igual a 1.6
Índice de endeudamiento Menor o igual al 70 %
Razón de cobertura de intereses Mayor o igual a 2.3 (o indeterminado)
Capital de trabajo Mayor o igual al 20 % del presupuesto oficial
Rentabilidad del patrimonio Mayor o igual a 0.03
Rentabilidad del activo Mayor o igual a 0.03
Resultado fuera del corpus: []


## 3. MCP y VS Code

El cliente oficial inicia `secop_mcp.server` por **stdio**, descubre el contrato y calcula indicadores a partir del perfil empresarial. El servidor admite modo muestra y API; esta demostración fija **muestra** para reproducibilidad.

En VS Code: abrir la raíz, revisar `.vscode/mcp.json`, iniciar el servidor `secop` desde la configuración MCP y abrir Copilot Chat en modo **Agent**. Habilitar sus cuatro herramientas y pedir, por ejemplo, «Calcula la liquidez de la empresa y cita el archivo de origen». El servidor `memory` configurado para Copilot es una herramienta de memoria distinta de las tres capas Chroma del agente Python; no comparte automáticamente sus hechos.

In [3]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
parameters = StdioServerParameters(command=sys.executable, args=["-m", "secop_mcp.server"],
                                  cwd=str(PROJECT_ROOT), env={"MODO_DATOS": "muestra"})
async with stdio_client(parameters) as (reader, writer):
    async with ClientSession(reader, writer) as session:
        await session.initialize()
        tools = await session.list_tools()
        print("MCP tools:", [tool.name for tool in tools.tools])
        response = await session.call_tool("compute_financial_indicators", {"process_budget": 600000000})
        payload = json.loads(response.content[0].text)
        print(json.dumps(payload, ensure_ascii=False, indent=2))
        print(f"[fuente: {payload['source']}]")
print((PROJECT_ROOT / ".vscode" / "mcp.json").read_text())


MCP tools: ['search_processes', 'get_process', 'get_company_profile', 'compute_financial_indicators']
{
  "liquidity": 2.101156069364162,
  "debt_ratio": 0.55,
  "interest_coverage": 3.2,
  "working_capital": 381000000.0,
  "roe": 0.15555555555555556,
  "roa": 0.07,
  "financial_statement_date": "2025-12-31",
  "currency": "COP",
  "source": "datos/empresa_multiproposito.json",
  "working_capital_pct_of_budget": 63.5
}
[fuente: datos/empresa_multiproposito.json]
{
  "servers": {
    "secop": {
      "type": "stdio",
      "command": "uv",
      "args": ["--directory", "${workspaceFolder}", "run", "secop-mcp"]
    },
    "memory": {
      "type": "stdio",
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-memory"],
      "env": {
        "MEMORY_FILE_PATH": "${workspaceFolder}/memoria/memory.jsonl"
      }
    }
  }
}



## 4. Agente con citas y abstención

Claude es el modelo principal. Las herramientas locales devuelven texto y etiquetas exactas; MCP devuelve `source`. El guard valida la procedencia de las citas del turno y sustituye respuestas sin evidencia por «No lo sé». Una etiqueta válida no demuestra por sí sola que cada frase esté sustentada: la evaluación mide también corrección y fuente esperada.

In [4]:
from secop_agent.agent import SecopAssistant
os.environ["MODO_DATOS"] = "muestra"
demo_questions = [
    "¿Cuál es el índice de liquidez mínimo de FDLS-LP-098-2019?",
    "¿En cuántas cuotas mensuales se paga el contrato de 02_CD_55_2021_3516.pdf?",
    "¿Cuáles son los códigos UNSPSC de experiencia de la ALCALDIA DE FLORIDABLANCA?",
    "Calcula el capital de trabajo y la liquidez de nuestra empresa con la herramienta de indicadores.",
    "Consulta en SECOP el presupuesto del proceso FDLS-LP-098-2019.",
]
missing_questions = [
    "¿Cuál es la prima del seguro de vida del gerente en el proceso 11?",
    "¿Cuál es el correo electrónico del supervisor de EDP-545-2022?",
    "¿A qué distancia está la Luna de Júpiter?",
]
demo_outputs = []
with TemporaryDirectory(prefix="secop-demo-") as directory:
    async with SecopAssistant(memory_dir=directory, user_id="demo") as assistant:
        for question in demo_questions + missing_questions:
            result = await assistant.ask(question)
            demo_outputs.append(asdict(result))
            print("\nPregunta:", question)
            print(result.answer)
            print("Citas:", result.citations)
            print("Métricas:", result.metrics, "run_id:", result.run_id)
            if question in missing_questions:
                assert result.abstained and result.answer.startswith("No lo sé")
            else:
                assert result.citations, "An answerable demo must cite evidence"
    print("Coste sesión, incluida consolidación:", assistant.session_metrics)



Pregunta: ¿Cuál es el índice de liquidez mínimo de FDLS-LP-098-2019?
El índice de liquidez mínimo exigido en FDLS-LP-098-2019 es 1.6 (índice de liquidez mayor o igual a 1.6). [fuente: 11_FDLS_LP_098_2019.pdf, p. 2, §5 Capacidad financiera y organizacional (según RUP)]
Citas: ['[fuente: 11_FDLS_LP_098_2019.pdf, p. 2, §5 Capacidad financiera y organizacional (según RUP)]']
Métricas: {'model_calls': 2, 'input_tokens': 5027, 'output_tokens': 203, 'cache_read_tokens': 0, 'cost_usd': 0.000604, 'cost_complete': True, 'latency_s': 4.3504207899968605} run_id: 3fe01055-c164-4c74-8e22-d93bca184a47



Pregunta: ¿En cuántas cuotas mensuales se paga el contrato de 02_CD_55_2021_3516.pdf?
El contrato de 02_CD_55_2021_3516.pdf se paga en 6 cuotas mensuales, previa presentación del informe de actividades, la certificación de cumplimiento del supervisor y el pago de aportes al sistema de seguridad social. [fuente: 02_CD_55_2021_3516.pdf, p. 2, §6 Valor y forma de pago]
Citas: ['[fuente: 02_CD_55_2021_3516.pdf, p. 2, §6 Valor y forma de pago]']
Métricas: {'model_calls': 2, 'input_tokens': 5386, 'output_tokens': 341, 'cache_read_tokens': 0, 'cost_usd': 0.000709, 'cost_complete': True, 'latency_s': 2.761194073002116} run_id: c6ec1af2-bd14-4a25-96d3-1b70948f6cc6



Pregunta: ¿Cuáles son los códigos UNSPSC de experiencia de la ALCALDIA DE FLORIDABLANCA?
Los códigos UNSPSC de experiencia exigidos por la ALCALDÍA DE FLORIDABLANCA en el proceso FLO-INFRA-SA-MC-007-2025 son 72101500 y 72152700. Se exige acreditar máximo 4 contratos inscritos en el RUP clasificados en al menos uno de esos códigos, cuya suma sea igual o superior al 100 % del presupuesto oficial en SMMLV, y al menos uno con valor igual o superior al 50 % del presupuesto oficial. [fuente: 14_FLO_INFRA_SA_MC_007_2025.pdf, p. 2, §6 Experiencia del proponente]
Citas: ['[fuente: 14_FLO_INFRA_SA_MC_007_2025.pdf, p. 2, §6 Experiencia del proponente]']
Métricas: {'model_calls': 5, 'input_tokens': 22022, 'output_tokens': 1251, 'cache_read_tokens': 0, 'cost_usd': 0.002828, 'cost_complete': True, 'latency_s': 11.498576050002157} run_id: a621e862-ba2c-41a6-a443-ef89e6ccf34a



Pregunta: Calcula el capital de trabajo y la liquidez de nuestra empresa con la herramienta de indicadores.
El capital de trabajo de la empresa es de COP 381.000.000 y su índice de liquidez es de 2,10, con estados financieros al 31 de diciembre de 2025. [fuente: datos/empresa_multiproposito.json]
Citas: ['[fuente: datos/empresa_multiproposito.json]']
Métricas: {'model_calls': 2, 'input_tokens': 6428, 'output_tokens': 250, 'cache_read_tokens': 0, 'cost_usd': 0.000768, 'cost_complete': True, 'latency_s': 3.966657552002289} run_id: f24133d9-e748-4d88-af2b-896bf9cfb81d



Pregunta: Consulta en SECOP el presupuesto del proceso FDLS-LP-098-2019.
El presupuesto (precio base) del proceso FDLS-LP-098-2019 es de COP 600.000.000. [fuente: datos/secop_muestra.json]

Corrección sobre la respuesta anterior: no había consultado la herramienta en ese turno. Al volver a consultarla, confirmo que el capital de trabajo es de COP 381.000.000 y el índice de liquidez de 2,10, con estados financieros al 31 de diciembre de 2025. [fuente: datos/empresa_multiproposito.json]
Citas: ['[fuente: datos/secop_muestra.json]', '[fuente: datos/empresa_multiproposito.json]']
Métricas: {'model_calls': 2, 'input_tokens': 7560, 'output_tokens': 676, 'cache_read_tokens': 0, 'cost_usd': 0.001094, 'cost_complete': True, 'latency_s': 5.4355099919994245} run_id: cf44ef2e-2b86-4406-be56-441160c4ff3f



Pregunta: ¿Cuál es la prima del seguro de vida del gerente en el proceso 11?
No lo sé. No encontré en los documentos indexados información sobre una prima de seguro de vida del gerente. Busqué en el proceso 11, que corresponde al archivo 11_FDLS_LP_098_2019.pdf (FDLS-LP-098-2019) [fuente: 11_FDLS_LP_098_2019.pdf], con los términos "prima seguro de vida gerente" y "seguro de vida", y ambas búsquedas no arrojaron resultados.
Citas: ['[fuente: 11_FDLS_LP_098_2019.pdf]']
Métricas: {'model_calls': 6, 'input_tokens': 25076, 'output_tokens': 1853, 'cache_read_tokens': 0, 'cost_usd': 0.003434, 'cost_complete': True, 'latency_s': 12.929665346000547} run_id: 0b8b4e5b-0c9d-4506-8fc7-5ad27abc0c95



Pregunta: ¿Cuál es el correo electrónico del supervisor de EDP-545-2022?
No lo sé. En los documentos indexados del proceso EDP-545-2022 no encontré el correo electrónico del supervisor. La búsqueda de "correo electrónico supervisor" y de "correo" no arrojó resultados. La única mención al supervisor es la de la forma de pago, que solo indica que el pago requiere la certificación de cumplimiento del supervisor [fuente: 01_EDP_545_2022.pdf, p. 2, §6 Valor y forma de pago].
Citas: ['[fuente: 01_EDP_545_2022.pdf, p. 2, §6 Valor y forma de pago]']
Métricas: {'model_calls': 5, 'input_tokens': 17440, 'output_tokens': 934, 'cache_read_tokens': 0, 'cost_usd': 0.002211, 'cost_complete': True, 'latency_s': 9.465924959000404} run_id: 74dcd1da-2c3d-4e8b-83f8-d3ca6f1477e2



Pregunta: ¿A qué distancia está la Luna de Júpiter?
No lo sé. Solo puedo responder con los documentos de licitación indexados, los datos de la empresa y la memoria del usuario, y ninguno contiene información sobre la distancia entre la Luna y Júpiter.
Citas: []
Métricas: {'model_calls': 2, 'input_tokens': 4677, 'output_tokens': 852, 'cache_read_tokens': 0, 'cost_usd': 0.000894, 'cost_complete': True, 'latency_s': 4.730804251001246} run_id: 312e2112-bc4c-4e0d-b4f3-8f70484cf96e


Coste sesión, incluida consolidación: {'model_calls': 28, 'input_tokens': 95919, 'output_tokens': 7252, 'cache_read_tokens': 0, 'cost_usd': 0.013218, 'cost_complete': True}


## 5. Memoria entre sesiones

Corto plazo: ventana de mensajes; medio plazo: resúmenes de sesiones con retención de 14 días; largo plazo: hechos personales, deduplicados por embeddings y con importancia/recencia. Las llamadas de resumen y extracción usan **el mismo modelo del asistente**.

La primera sesión registra nombre, ocupación, preferencias e interés. La segunda crea un objeto nuevo con el mismo directorio y usuario; la continuidad procede del disco.

`/memoria` muestra `snapshot()`. `/olvida <texto>` llama a `forget(text)` y elimina hechos cercanos de largo plazo. **No es un borrado integral:** el texto puede seguir en resúmenes o ventana de mensajes. El notebook muestra esta limitación y no promete olvido total.

**Observación de la ejecución guardada:** la consulta inicial combinó presentación personal y liquidez y recibió una abstención, aunque el dato existe. La prueba demuestra persistencia de los cuatro datos personales, no acierto en esa consulta compuesta. El resumen conservó esa respuesta fallida: otro motivo para tratar memoria como contexto falible.

In [5]:
memory_directory = PROJECT_ROOT / "memoria" / "notebook-demo"
from uuid import uuid4
memory_user = "notebook-" + uuid4().hex
async with SecopAssistant(memory_dir=memory_directory, user_id=memory_user) as first_session:
    introduction = ("Me llamo Valeria, trabajo en una constructora de Bogotá, prefiero respuestas breves "
                    "con citas y me interesa FDLS-LP-098-2019. ¿Cuál es su índice de liquidez mínimo?")
    print((await first_session.ask(introduction)).answer)
print("Sesión 1 cerrada. Uso total:", first_session.session_metrics)
async with SecopAssistant(memory_dir=memory_directory, user_id=memory_user) as second_session:
    recalled = await second_session.ask("¿Cómo me llamo, dónde trabajo, qué respuestas prefiero y qué proceso me interesa?")
    print("Nueva instancia:", recalled.answer)
    assert "[fuente: memoria]" in recalled.citations
    snapshot_before = await asyncio.to_thread(second_session.memory.snapshot)
    print("/memoria:", json.dumps(snapshot_before, ensure_ascii=False, indent=2))
    assert snapshot_before["facts"] and snapshot_before["sessions"]
    fact_to_forget = next((item["fact"] for item in snapshot_before["facts"] if "prefier" in item["fact"].lower()), snapshot_before["facts"][0]["fact"])
    print("/olvida", fact_to_forget)
    forgotten = await asyncio.to_thread(second_session.memory.forget, fact_to_forget)
    print("Hechos eliminados:", forgotten)
    assert forgotten >= 1
    print("Después:", json.dumps(await asyncio.to_thread(second_session.memory.snapshot), ensure_ascii=False, indent=2))


Valeria, el índice de liquidez mínimo exigido en FDLS-LP-098-2019 es **mayor o igual a 1.6**. [fuente: 11_FDLS_LP_098_2019.pdf, p. 2, §5 Capacidad financiera y organizacional (según RUP)]


Sesión 1 cerrada. Uso total: {'model_calls': 4, 'input_tokens': 6078, 'output_tokens': 578, 'cache_read_tokens': 0, 'cost_usd': 0.000897, 'cost_complete': True}


Nueva instancia: Te llamas Valeria [fuente: memoria]. Trabajas en una constructora de Bogotá [fuente: memoria]. Prefieres respuestas breves con citas de fuente [fuente: memoria]. El proceso que te interesa es FDLS-LP-098-2019 [fuente: memoria].
/memoria: {
  "facts": [
    {
      "id": "fact-8a9c668d35f246b78f2b0fbc42a225d8",
      "fact": "La usuaria trabaja en una constructora de Bogotá.",
      "importance": 8,
      "created_at": 1793221481.702611,
      "last_access": 1793221483.078952
    },
    {
      "id": "fact-45ca618eb8bd40e284d8d4372c712bc6",
      "fact": "La usuaria prefiere respuestas breves y con citas de fuente.",
      "importance": 8,
      "created_at": 1793221481.762723,
      "last_access": 1793221483.078952
    },
    {
      "id": "fact-25ffc559d5c24ce192434fdc432f20ef",
      "fact": "La usuaria da seguimiento al proceso FDLS-LP-098-2019.",
      "importance": 8,
      "created_at": 1793221481.824082,
      "last_access": 1793221483.078952
    },
    {
      

## 6. Evaluación reproducible

Se ejecutan los scripts con checkpoints ya medidos si existen. **B-Claude:** prompt + guard; **A-Claude:** prompt sin guard; **C-local:** `ollama:qwen2.5:3b` + guard. Cada pregunta usa un usuario independiente y memoria temporal; ningún caso aprende la respuesta de otro.

Juez: salida estructurada del mismo Claude económico, sin parámetros de muestreo. El coste del juez figura separado, incluso al juzgar Ollama. La fuente acertada exige el PDF esperado en una etiqueta; no basta mencionar su nombre en prosa. Fallos operativos cuentan como incorrectos; no como abstenciones correctas. Precisión de extracción: igualdad en 280 campos (20 × 14), sin consultar la clave al extraer. Ollama local no tiene precio por token; energía y hardware no están estimados.

La subida a LangSmith reutiliza las salidas medidas y no repite el agente. Los `run_id` de cada pregunta apuntan a las trazas originales.

In [6]:
import importlib.util

def load_script(name):
    spec = importlib.util.spec_from_file_location(name, PROJECT_ROOT / "scripts" / f"{name}.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

from argparse import Namespace
rag_evaluation = load_script("evaluate")
rag_report = await rag_evaluation.run(Namespace(models=[settings.chat_model, "ollama:qwen2.5:3b"],
    judge_model=settings.chat_model, limit=None, timeout=180, force=False, no_publish=False))
extraction_evaluation = load_script("evaluate_extraction")
extraction_report = await asyncio.to_thread(extraction_evaluation.evaluate, [settings.chat_model])
display(Markdown(rag_evaluation.render_report(rag_report).split("## Detalle")[0]))
display(Markdown((RESULTS_DIR / "extraccion_requisitos.md").read_text()))
print("LangSmith:", rag_report["langsmith"])
from langsmith import Client
client = Client()
project = client.read_project(project_name=settings.langsmith_project)
project_url = getattr(project, "url", None)
print("Proyecto LangSmith:", project_url or settings.langsmith_project)


View the evaluation results for experiment: 'B-claude-83af1cf3' at:
https://smith.langchain.com/o/be862a28-1dbf-4dec-b6c4-f58119a8f835/datasets/acf6ac2c-1ee2-42dd-8b9a-f39e61411058/compare?selectedSessions=e41286b7-7a90-462c-abe4-76b22f2a3dd5




0it [00:00, ?it/s]

View the evaluation results for experiment: 'A-claude-815314f1' at:
https://smith.langchain.com/o/be862a28-1dbf-4dec-b6c4-f58119a8f835/datasets/acf6ac2c-1ee2-42dd-8b9a-f39e61411058/compare?selectedSessions=18a87fd3-666a-4776-8cd3-da185f7fcb14




0it [00:00, ?it/s]

View the evaluation results for experiment: 'C-local-20af14b1' at:
https://smith.langchain.com/o/be862a28-1dbf-4dec-b6c4-f58119a8f835/datasets/acf6ac2c-1ee2-42dd-8b9a-f39e61411058/compare?selectedSessions=562a1b79-5655-4d27-9db7-cf6574910271




0it [00:00, ?it/s]

# Evaluación RAG + MCP

Corpus educativo simulado. Juez LLM; una ejecución por pregunta, sin intervalos de confianza.
Fuentes: coincidencia del PDF citado (no verifica implicación semántica). Abstención: 8 ausentes; falsas abstenciones y fuentes: 16 respondibles.

| Configuración | n | Corrección | Fuente | Abstención | Falsa abstención | USD agente + memoria | USD juez | Latencia media s | Errores |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| B-claude | 24 | 95.8% | 93.8% | 100.0% | 0.0% | 0.035452 | 0.003996 | 5.04 | 0 |
| A-claude | 24 | 91.7% | 81.2% | 87.5% | 6.2% | 0.036906 | 0.003826 | 5.16 | 1 |
| C-local | 24 | 33.3% | 0.0% | 100.0% | 87.5% | 0.000000 | 0.003683 | 2.23 | 0 |

La latencia incluye recuperación y registro del turno; consolidar al cierre se mide aparte. Costes según secop_agent.costs; Ollama: precio por token cero, sin estimar energía ni hardware.



# Extracción de requisitos

20 PDF simulados × 14 campos = 280 comparaciones. Tolerancia numérica absoluta: 1e-6; listas sin orden; ausentes cuentan como error.

| Modelo | Aciertos | Total | Exactitud | USD | Tokens entrada | Tokens salida | Tiempo s |
|---|---:|---:|---:|---:|---:|---:|---:|
| anthropic:claude-haiku-5-5 | 279 | 280 | 99.6% | 0.009803 | 55865 | 8435 | 116.19 |

## anthropic:claude-haiku-5-5

| Campo | Aciertos / 20 | Exactitud |
|---|---:|---:|
| is_direct_contracting | 20 / 20 | 100.0% |
| is_sale_of_goods | 20 / 20 | 100.0% |
| verifies_financial_capacity | 20 / 20 | 100.0% |
| min_liquidity | 20 / 20 | 100.0% |
| max_debt_ratio_pct | 20 / 20 | 100.0% |
| min_interest_coverage | 20 / 20 | 100.0% |
| min_working_capital_budget_pct | 20 / 20 | 100.0% |
| min_return_on_equity | 20 / 20 | 100.0% |
| min_return_on_assets | 20 / 20 | 100.0% |
| experience_unspsc_codes | 20 / 20 | 100.0% |
| experience_by_similar_object | 19 / 20 | 95.0% |
| max_experience_contracts | 20 / 20 | 100.0% |
| experience_budget_pct | 20 / 20 | 100.0% |
| min_single_contract_budget_pct | 20 / 20 | 100.0% |

| PDF | Aciertos / 14 |
|---|---:|
| 01_EDP_545_2022.pdf | 14 / 14 |
| 02_CD_55_2021_3516.pdf | 13 / 14 |
| 03_RE_CGBM_049_2024.pdf | 14 / 14 |
| 04_CD_CPS_0720_2026.pdf | 14 / 14 |
| 05_PS_147_2025.pdf | 14 / 14 |
| 06_SINF_CPS_050_23.pdf | 14 / 14 |
| 07_135116.pdf | 14 / 14 |
| 08_OC10_2494_2025.pdf | 14 / 14 |
| 09_SDIS_CPS_302405_2024.pdf | 14 / 14 |
| 10_009_2021.pdf | 14 / 14 |
| 11_FDLS_LP_098_2019.pdf | 14 / 14 |
| 12_FNV_001_2021.pdf | 14 / 14 |
| 13_LP_DT_CAU_003_2019.pdf | 14 / 14 |
| 14_FLO_INFRA_SA_MC_007_2025.pdf | 14 / 14 |
| 15_SA_004_2023.pdf | 14 / 14 |
| 16_CD_PS_GDCH_186SS_2024.pdf | 14 / 14 |
| 17_5308_SASP01_2023.pdf | 14 / 14 |
| 18_0511_2024.pdf | 14 / 14 |
| 19_CSS_297_2024.pdf | 14 / 14 |
| 20_CPS_009_2026.pdf | 14 / 14 |

Extracción local: Skipped: optional local extraction deferred to keep runtime bounded.

Coste según la tabla del proyecto. No se consulta la clave durante la extracción.


LangSmith: {'dataset': 'secop-reto2-pliegos', 'experiments': [{'configuration': 'B-claude', 'experiment': 'B-claude-83af1cf3'}, {'configuration': 'A-claude', 'experiment': 'A-claude-815314f1'}, {'configuration': 'C-local', 'experiment': 'C-local-20af14b1'}]}


Proyecto LangSmith: https://smith.langchain.com/o/be862a28-1dbf-4dec-b6c4-f58119a8f835/projects/p/6f480c7b-d861-4a8b-af41-8211bf4a1efc


## 7. Conclusiones y decisiones

La recomendación se deriva de las mediciones siguientes; no se infiere calidad por tamaño o proveedor. Los resultados tienen alcance académico: corpus simulado, 24 preguntas, una ejecución por configuración y un juez LLM. El fixture también calibró el retriever, por lo que una nota alta no demuestra rendimiento en documentos nuevos.

El guard comprueba procedencia, no implicación semántica por frase; el prompt exige la cita en cada frase factual. Las respuestas negativas mantienen el prefijo «No lo sé». La memoria es contexto personal y se cita como tal; sus resúmenes pueden contener errores. La separación MCP permite consumir las mismas herramientas desde Python y Copilot sin entregar al servidor el control del agente.

**Elección con datos:** B-Claude obtuvo 24/24 correctas; A-Claude 23/24 (un límite de pasos); C-local 8/24. Se recomienda B-Claude para la entrega. Fuente esperada: 15/16 para ambos Claude y 0/16 para Qwen.

**Auditoría del juez:** C-local/q22 recibió un segundo juicio porque el veredicto original contradecía su explicación sobre «No lo sé». No se repitió el agente; se conserva el juicio inicial en JSON y se suman ambos costes. Por ello la evaluación incluye una revisión dirigida y no es completamente automática.

In [7]:
ranked = sorted(rag_report["results"], key=lambda result: (result["overall"]["correctness"] or 0, result["overall"]["source_accuracy"] or 0), reverse=True)
for result in ranked:
    score = result["overall"]
    print(result["configuration"], {key: score[key] for key in ("correctness", "source_accuracy", "abstention_accuracy", "false_abstention_rate", "cost_usd", "mean_latency_s")})
print("Mejor corrección observada:", ranked[0]["configuration"])
print("Decisiones detalladas:", PROJECT_ROOT / "docs" / "DESIGN_DECISIONS.md")


B-claude {'correctness': 0.9583333333333334, 'source_accuracy': 0.9375, 'abstention_accuracy': 1.0, 'false_abstention_rate': 0.0, 'cost_usd': 0.035452, 'mean_latency_s': 5.037036085791745}
A-claude {'correctness': 0.9166666666666666, 'source_accuracy': 0.8125, 'abstention_accuracy': 0.875, 'false_abstention_rate': 0.0625, 'cost_usd': 0.036906, 'mean_latency_s': 5.1558624622916795}
C-local {'correctness': 0.3333333333333333, 'source_accuracy': 0.0, 'abstention_accuracy': 1.0, 'false_abstention_rate': 0.875, 'cost_usd': 0.0, 'mean_latency_s': 2.2292808831254356}
Mejor corrección observada: B-claude
Decisiones detalladas: /home/esteban/Documents/University/Tenth semester/IA/unit-iii/challenge-ii/reto2-secop-rag-mcp/docs/DESIGN_DECISIONS.md
